In [14]:
import cv2
import mediapipe as mp
import numpy as np

mp_face_mesh = mp.solutions.face_mesh

SUBNASALE   = 2
UPPER_LIP   = 13
LOWER_LIP   = 14
CHIN_BOTTOM = 152
NOSE_TIP    = 1
FACE_EDGE_R = 454

def draw_dashed_line(img, pt1, pt2, color, thickness=1, dash=6, gap=4):
    dist = int(np.hypot(pt2[0]-pt1[0], pt2[1]-pt1[1]))
    if dist == 0:
        return
    dx, dy = (pt2[0]-pt1[0])/dist, (pt2[1]-pt1[1])/dist
    i = 0
    while i < dist:
        s = (int(pt1[0]+dx*i), int(pt1[1]+dy*i))
        e = (int(pt1[0]+dx*min(i+dash, dist)), int(pt1[1]+dy*min(i+dash, dist)))
        cv2.line(img, s, e, color, thickness)
        i += dash + gap

def analyze_profile(img_path, out_path="out.jpg"):
    img = cv2.imread(img_path)
    h, w = img.shape[:2]
    color = (255, 255, 255)  # white, all lines

    with mp_face_mesh.FaceMesh(static_image_mode=True, max_num_faces=1,
                                refine_landmarks=True) as fm:
        res = fm.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        if not res.multi_face_landmarks:
            print("no face found")
            return
        lm = res.multi_face_landmarks[0].landmark

        def pt(idx):
            return (int(lm[idx].x * w), int(lm[idx].y * h))

        p_subnasale = pt(SUBNASALE)
        p_upper_lip = pt(UPPER_LIP)
        p_lower_lip = pt(LOWER_LIP)
        p_chin      = pt(CHIN_BOTTOM)
        p_edge      = pt(FACE_EDGE_R)

        xs = [p_subnasale[0], p_upper_lip[0], p_lower_lip[0], p_chin[0], pt(NOSE_TIP)[0]]
        x_vert = max(xs) + 25
        x_left = min(xs) - 15
        x_right_horiz = x_vert

        for p in [p_subnasale, p_upper_lip, p_lower_lip, p_chin]:
            draw_dashed_line(img, (x_left, p[1]), (x_right_horiz, p[1]), color)

        y_top = p_subnasale[1] - 10
        y_bot = p_chin[1] + 10
        draw_dashed_line(img, (x_vert, y_top), (x_vert, y_bot), color)

        for p in [p_subnasale, p_upper_lip, p_lower_lip, p_chin]:
            cv2.circle(img, p, 3, color, -1)

    cv2.imwrite(out_path, img)
    print(f"saved {out_path}")

analyze_profile(r"C:\Users\JayRabari\Documents\FacialAnalysis\var\media\assessments\cf9cf21f-5f65-4160-bd88-c6160af6adfb\rightProfile.jpg")

saved out.jpg


In [1]:
import cv2
import mediapipe as mp
import numpy as np

mp_face_mesh = mp.solutions.face_mesh

CHIN_BOTTOM = 152
NOSE_TIP    = 1

def draw_dashed_line(img, pt1, pt2, color, thickness=1, dash=6, gap=4):
    dist = int(np.hypot(pt2[0]-pt1[0], pt2[1]-pt1[1]))
    if dist == 0:
        return
    dx, dy = (pt2[0]-pt1[0])/dist, (pt2[1]-pt1[1])/dist
    i = 0
    while i < dist:
        s = (int(pt1[0]+dx*i), int(pt1[1]+dy*i))
        e = (int(pt1[0]+dx*min(i+dash, dist)), int(pt1[1]+dy*min(i+dash, dist)))
        cv2.line(img, s, e, color, thickness)
        i += dash + gap

def annotate_region(img_path, out_path="out1.jpg"):
    img = cv2.imread(img_path)
    h, w = img.shape[:2]
    color = (255, 255, 255)

    with mp_face_mesh.FaceMesh(static_image_mode=True, max_num_faces=1,
                                refine_landmarks=True) as fm:
        res = fm.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        if not res.multi_face_landmarks:
            print("no face found")
            return
        lm = res.multi_face_landmarks[0].landmark

        def pt(idx):
            return (int(lm[idx].x * w), int(lm[idx].y * h))

        p_chin = pt(CHIN_BOTTOM)
        p_nose = pt(NOSE_TIP)

        draw_dashed_line(img, p_chin, p_nose, color)

    cv2.imwrite(out_path, img)
    print(f"saved {out_path}")

annotate_region(
    r"C:\Users\JayRabari\Documents\FacialAnalysis\var\media\assessments\cf9cf21f-5f65-4160-bd88-c6160af6adfb\right45.jpg"
)

saved out1.jpg


c:\Users\JayRabari\Documents\FacialAnalysis\.venv\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


In [2]:
import cv2
import mediapipe as mp
import numpy as np

def draw_dashed_line(img, pt1, pt2, color, thickness=1, dash=6, gap=4):
    dist = int(np.hypot(pt2[0]-pt1[0], pt2[1]-pt1[1]))
    if dist == 0:
        return
    dx, dy = (pt2[0]-pt1[0])/dist, (pt2[1]-pt1[1])/dist
    i = 0
    while i < dist:
        s = (int(pt1[0]+dx*i), int(pt1[1]+dy*i))
        e = (int(pt1[0]+dx*min(i+dash, dist)), int(pt1[1]+dy*min(i+dash, dist)))
        cv2.line(img, s, e, color, thickness)
        i += dash + gap

def annotate_profile(img_path, out_path_top="out1.jpg", out_path_bottom="out2.jpg"):
    image = cv2.imread(img_path)
    if image is None:
        print("could not load image")
        return
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    mp_face_mesh = mp.solutions.face_mesh
    with mp_face_mesh.FaceMesh(static_image_mode=True, max_num_faces=1,
                                refine_landmarks=True,
                                min_detection_confidence=0.2) as face_mesh:
        results = face_mesh.process(image_rgb)

        if not results.multi_face_landmarks:
            print("No face detected. Please ensure the side profile image is clear.")
            return

        landmarks = results.multi_face_landmarks[0].landmark
        h, w, _ = image.shape

        def get_pt(idx):
            return (int(landmarks[idx].x * w), int(landmarks[idx].y * h))

        nose_tip  = get_pt(4)
        subnasale = get_pt(2)
        nasion    = get_pt(168)
        upper_lip = get_pt(0)
        chin      = get_pt(152)

        line_color = (255, 255, 255)
        thickness = 2

        img_top = image.copy()
        img_bottom = image.copy()

        # Style 1: nose projection lines
        cv2.line(img_top, (nose_tip[0], subnasale[1]), (nose_tip[0], chin[1]), line_color, thickness)
        draw_dashed_line(img_top, subnasale, (nose_tip[0], subnasale[1]), line_color, thickness)
        draw_dashed_line(img_top, upper_lip, (nose_tip[0], upper_lip[1]), line_color, thickness)
        draw_dashed_line(img_top, chin, (nose_tip[0], chin[1]), line_color, thickness)

        # Style 2: E-line and true verticals
        cv2.line(img_bottom, nose_tip, chin, line_color, thickness)
        cv2.line(img_bottom, nasion, (nasion[0], chin[1] + 20), line_color, thickness)
        cv2.line(img_bottom, subnasale, (subnasale[0], chin[1] + 20), line_color, thickness)

    cv2.imwrite(out_path_top, img_top)
    cv2.imwrite(out_path_bottom, img_bottom)
    print(f"saved {out_path_top} and {out_path_bottom}")

annotate_profile(
    r"C:\Users\JayRabari\Documents\FacialAnalysis\var\media\assessments\cf9cf21f-5f65-4160-bd88-c6160af6adfb\right45.jpg",
    out_path_top="out1.jpg",
    out_path_bottom="out2.jpg"
)

saved out1.jpg and out2.jpg


c:\Users\JayRabari\Documents\FacialAnalysis\.venv\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
